# AI 기본법 조문·항 단위 청킹 Qdrant 테스트

조문 전체(article)와 항(paragraph)을 각각 청크로 만들어 Qdrant에 저장합니다.

In [1]:
from pathlib import Path
import re
import xml.etree.ElementTree as ET

from common.ai_model import get_embedding_model
from common.qdrant import get_qdrant_client
from qdrant_client.models import Distance, PointStruct, VectorParams

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
XML_PATH = PROJECT_ROOT / 'data' / 'ai_basic_law' / 'ai_basic_law.xml'
if not XML_PATH.exists():
    raise FileNotFoundError(f'XML 파일이 없습니다: {XML_PATH}')

root = ET.parse(XML_PATH).getroot()
print(f'XML 로드 완료: {XML_PATH}')

XML 로드 완료: /home/aiuser/rag_mini_pjt_template/data/ai_basic_law/ai_basic_law.xml


In [2]:
def clean_text(element):
    return ' '.join(''.join(element.itertext()).split())

chunks = []
SECTION_ONLY = re.compile(r'^제\d+(편|장|절|관|항)\s+')

for unit in root.findall('.//조문단위'):
    article_no = unit.findtext('조문번호', '').strip()
    article_title = unit.findtext('조문제목', '').strip()
    article_content = clean_text(unit)

    # 장·절 제목만 있는 구조 노드는 제외
    if SECTION_ONLY.match(article_content) and not article_title:
        continue

    # 조문 전체 청크
    article_parts = []
    for element in unit:
        if element.tag in {'조문내용', '항내용', '호내용', '목내용'}:
            text = clean_text(element)
            if text and text not in article_parts:
                article_parts.append(text)
    article_text = '\n'.join(article_parts)
    if article_text:
        chunks.append({
            'id': len(chunks) + 1,
            'chunk_type': 'article',
            'article': f'제{article_no}조',
            'paragraph': '',
            'title': article_title,
            'content': article_text,
            'source': XML_PATH.name,
        })

    # 항 단위 청크
    paragraph_no = 0
    for paragraph in unit.findall('./항'):
        paragraph_no += 1
        paragraph_text = clean_text(paragraph)
        if not paragraph_text:
            continue
        chunks.append({
            'id': len(chunks) + 1,
            'chunk_type': 'paragraph',
            'article': f'제{article_no}조',
            'paragraph': f'{paragraph_no}항',
            'title': article_title,
            'content': paragraph_text,
            'source': XML_PATH.name,
        })

print(f'조문·항 청크 수: {len(chunks)}개')
print(chunks[0] if chunks else '청크가 없습니다.')

조문·항 청크 수: 220개
{'id': 1, 'chunk_type': 'article', 'article': '제1조', 'paragraph': '', 'title': '', 'content': '제1장 총칙', 'source': 'ai_basic_law.xml'}


In [3]:
embedding_client = get_embedding_model()
qdrant_client = get_qdrant_client()
COLLECTION_NAME = 'ai_basic_law_article_paragraph_v1'

vectors = await embedding_client.aembed_documents([chunk['content'] for chunk in chunks])

if qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.delete_collection(COLLECTION_NAME)

qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=len(vectors[0]), distance=Distance.COSINE),
)

points = [
    PointStruct(id=chunk['id'], vector=vector, payload=chunk)
    for chunk, vector in zip(chunks, vectors)
]
qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points)
print(f'{len(points)}개 청크를 Qdrant에 저장했습니다.')

220개 청크를 Qdrant에 저장했습니다.


In [5]:
query = '고영향 인공지능이란 무엇인가요?'
query_vector = await embedding_client.aembed_query(query)
result = qdrant_client.query_points(collection_name=COLLECTION_NAME, query=query_vector, limit=5)

for point in result.points:
    payload = point.payload
    print(f"[{payload['chunk_type']}] {payload['article']} {payload['paragraph']} {payload['title']} score={point.score:.4f}")
    print(payload['content'][:1500])
    print()

[paragraph] 제31조 1항 인공지능 투명성 확보 의무 score=0.6280
① ① 인공지능사업자는 고영향 인공지능이나 생성형 인공지능을 이용한 제품 또는 서비스를 제공하려는 경우 제품 또는 서비스가 해당 인공지능에 기반하여 운용된다는 사실을 이용자에게 사전에 고지하여야 한다.

[paragraph] 제34조 1항 고영향 인공지능과 관련한 사업자의 책무 score=0.5937
① ① 인공지능사업자는 고영향 인공지능 또는 이를 이용한 제품ㆍ서비스를 제공하는 경우 고영향 인공지능의 안전성ㆍ신뢰성을 확보하기 위하여 다음 각 호의 내용을 포함하는 조치를 대통령령으로 정하는 바에 따라 이행하여야 한다. 1. 1. 위험관리방안의 수립ㆍ운영 2. 2. 기술적으로 가능한 범위에서의 인공지능이 도출한 최종결과, 인공지능의 최종결과 도출에 활용된 주요 기준, 인공지능의 개발ㆍ활용에 사용된 학습용데이터의 개요 등에 대한 설명 방안의 수립ㆍ시행 3. 3. 이용자 보호 방안의 수립ㆍ운영 4. 4. 고영향 인공지능에 대한 사람의 관리ㆍ감독 5. 5. 안전성ㆍ신뢰성 확보를 위한 조치의 내용을 확인할 수 있는 문서의 작성과 보관 6. 6. 그 밖에 고영향 인공지능의 안전성ㆍ신뢰성 확보를 위하여 위원회에서 심의ㆍ의결된 사항

[paragraph] 제33조 1항 고영향 인공지능의 확인 score=0.5848
① ① 인공지능사업자는 인공지능 또는 이를 이용한 제품ㆍ서비스를 제공하는 경우 그 인공지능이 고영향 인공지능에 해당하는지에 대하여 사전에 검토하여야 하며, 필요한 경우 과학기술정보통신부장관에게 고영향 인공지능에 해당하는지 여부의 확인을 요청할 수 있다.

[paragraph] 제35조 1항 고영향 인공지능 영향평가 score=0.5835
① ① 인공지능사업자가 고영향 인공지능을 이용한 제품 또는 서비스를 제공하는 경우 사전에 사람의 기본권에 미치는 영향을 평가(이하 "영향평가"라 한다)하기 위하여 노력하여야 한다. 이 경우 영향평가에는 고영향 인공지능을 이용한 제품 또는 서비스의